# Lab 4a: Evaluating an Image Classifier

**Before you start:** If you are using Google Colab, remember to enable GPU.

This is **Part 1 of 3** in a mini-series on evaluating deep learning models:

| Part | Task | Focus |
|---|---|---|
| **4a (this notebook)** | Image classification | Build & train your own CNN, then evaluate it |
| 4b | Image segmentation | Use a pretrained model, evaluate with IoU & Dice |
| 4c | Object detection | Use a pretrained model, evaluate with IoU & mAP |

The three notebooks share one theme: **how do you know if your model is actually any good?**
Training a model is only half the job — a large part of doing (and reporting) deep learning work
is choosing the right evaluation metrics, computing them correctly, and interpreting what they tell you.

**Note:** This lab uses **PyTorch** instead of Keras. If you're used to Keras, the biggest differences you'll notice are:

- You write your own training loop (there is no `model.fit(...)`).
- PyTorch does not automatically infer layer input sizes — you need to work out (or empirically check) the shape flowing between layers yourself.
- Softmax is *not* part of the model — `nn.CrossEntropyLoss` expects raw, unnormalized scores (logits) and applies the softmax internally. You only apply softmax yourself when you want actual probabilities (e.g., for evaluation).

## Your task

1. Download and describe the [SVHN (Street View House Numbers)](http://ufldl.stanford.edu/housenumbers/) image classification dataset.
2. Design and describe a small CNN that can solve the SVHN problem.
3. Train your model and explain how you trained it.
4. Summarize your results using
   - Test set error (top-1 and top-5)
   - Test set accuracy
   - Confusion matrix
   - Precision/recall and Average Precision

We will be using PyTorch, [torchmetrics](https://lightning.ai/docs/torchmetrics/stable/) and optionally [scikit-learn](https://scikit-learn.org/stable/index.html).


In [ ]:
# Setup: install the one package Colab doesn't already include.
# (torch and torchvision are preinstalled on Colab; torchinfo gives us a Keras-style model.summary().)
!pip install -q torchinfo torchmetrics


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import transforms

import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

torch.manual_seed(0)  # make weight initialization (and other random ops) deterministic


## Task 1: The dataset
Describing your dataset is important. Here is a list of aspects that could be of interest to the reader:

- What are you trying to predict?
- Where did the dataset come from? (Remember to cite if it's a public dataset)
- How was it collected?
- Why was it collected?
- Why did you choose this dataset, and not that one over there?
- What is the output: categorical {0, 1, ..., K}, continuous scalar in [0, 1], arbitrary real no., etc.
- No. of classes, what are the classes?
- No. of observations, no. of observations per class if unbalanced.
- What is the size of the training set?
- Is there a test set? What is its size?
- Is there a baseline result that you can compare your results with?
- What is the state-of-the-art performance on this dataset?

Finally, it might also be a good idea to show some actual observations/examples from the dataset.

### 1.1 Download

In [ ]:
# The data, split between train and test sets.
# transforms.ToTensor() converts a PIL image (H, W, C) in [0, 255] to a torch.Tensor (C, H, W) in [0, 1].
# Note: SVHN needs `scipy` to load its .mat files -- already preinstalled on Colab.
train_dataset = torchvision.datasets.SVHN(root="./data", split="train", download=True, transform=transforms.ToTensor())
test_dataset = torchvision.datasets.SVHN(root="./data", split="test", download=True, transform=transforms.ToTensor())

# SVHN doesn't ship a `.classes` attribute -- it's just the 10 digits.
class_names = [str(i) for i in range(10)]
num_classes = len(class_names)
print("Classes:", class_names)
print("No. of training images:", len(train_dataset))
print("No. of test images:", len(test_dataset))


In [ ]:
# Show a few example images
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.permute(1, 2, 0))  # (C, H, W) -> (H, W, C) for matplotlib
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()
plt.show()


### 1.2 Questions
Try to answer as many of the above questions as possible (you don't need to write your answers down). You will be able to find many answers just by looking at the original source of the SVHN dataset:

http://ufldl.stanford.edu/housenumbers/

(SVHN images are 32×32 crops of house-number digits taken from Google Street View photos, so — unlike for instance CIFAR-10 — they're cropped from real, uncontrolled photographs rather than curated object photos. One quirk worth noting when you describe the dataset: because the crop is centered on one digit, a neighbouring digit is sometimes partially visible at the left/right edge of the image.)


## Task 2: Network Architecture
So you are faced with a machine learning problem. Which model should you use?

Well, that depends on the type of problem (classification, regression, clustering, etc.) and what the success criteria are (high accuracy, high speed, understanding structure in the data, etc).

### 2.1 Design a CNN
Your task is to design a CNN that solves the SVHN problem. Motivate your choice of architecture and hyperparameters (number of layers, number of neurons/kernels in each layer, etc.) and regularization techniques.

I left a template below that you could use — you just need to fill in the gaps (marked with `???`). Feel free to design your own network or search the internet for alternative models.

**Building blocks you have covered so far:** `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d`, `nn.Flatten`, `nn.Linear`. Stick to these for your main submission.

**A PyTorch-specific gotcha:** `nn.Linear` needs to know its exact input size up front — PyTorch won't infer it for you from the previous layer. In the template below, the two `MaxPool2d(2, 2)` layers and the one padding-free convolution together shrink a 32×32 input down to a 7×7 feature map, *regardless* of how many filters you choose (channels don't affect height/width). So the flattened vector length is `(number of filters in your last conv layer) × 7 × 7`. If you change the architecture (e.g. add another pooling layer), a quick way to check the size empirically is to run a dummy batch through just the convolutional part and print `.shape` — see the commented-out tip in the cell below.


In [ ]:
class ToyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, ???, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.Conv2d(???, ???, kernel_size=3),  # no padding ("valid"): 32x32 -> 30x30
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 30x30 -> 15x15
            nn.Conv2d(???, ???, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 15x15 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(??? * 7 * 7, ???),  # W1*x + b   (??? * 7 * 7 must match the last conv's filter count)
            nn.ReLU(),                    # ReLU(W1*x + b)
            nn.Linear(???, num_classes),  # W2*x + b  -- NOTE: no softmax here, CrossEntropyLoss adds it for you
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x  # raw logits


model = ToyCNN(num_classes=num_classes).to(device)

# Tip: to empirically check the flattened feature size instead of computing it by hand, try:
# dummy = torch.zeros(1, 3, 32, 32).to(device)  # batch size 1, 3 channels, 32x32 image
# print(model.features(dummy).shape)   # -> torch.Size([1, C, 7, 7])


**My model choice:** I chose a small capacity model over a large model, simply because it's faster to train. You can add more capacity by adding more neurons in each layer and/or adding more layers. What is the risk of increasing your model's capacity (i.e., number of weights)?

Always remember to summarize your model. In PyTorch we implement our own custom summarizer using forward hooks (see `lecture4_shape_comparison.ipynb`), or we can use third-party packages, like `torchinfo` (installed at the top of this notebook):

In [ ]:
from torchinfo import summary

summary(model, input_size=(1, 3, 32, 32))


### 2.2 Going further (optional): Batch Normalization & better optimizers

We haven't covered **batch normalization**, **dropout**, or optimizers beyond plain SGD (like **Adam**) in the lectures yet — they're coming up soon. But if you're curious, they are easy to try in PyTorch and often make a noticeable difference:

- **Batch normalization** (`nn.BatchNorm2d(num_features)`) is typically inserted right after a `Conv2d` and before its activation function. Try adding one after each conv layer in a copy of your model and see how it affects training speed and stability.
- **Adam** (`torch.optim.Adam(model.parameters(), lr=...)`) is a drop-in replacement for `torch.optim.SGD` in the training loop you'll write in Task 3. It often converges faster and is less sensitive to the learning rate than plain SGD.
- **Dropout** (`nn.Dropout(p=...)`) is a regularization technique often used after fully connected (linear) layers. Try adding it after the `ReLU` layer in the classifier and see how it affects your model's ability to generalize to unseen data.

This is entirely optional and won't be needed for the rest of this notebook — but feel free to duplicate the `ToyCNN` class below, add these two ingredients, and compare the resulting loss curves (Task 3) against your original model once you get there.

In [ ]:
# Optional playground -- not required for the rest of the notebook.
# Copy your ToyCNN above, add nn.BatchNorm2d(...) after each conv layer (before the ReLU),
# and try training it with optim.Adam instead of optim.SGD in Task 3.

# class ToyCNNWithBatchNorm(nn.Module):
#     def __init__(self, num_classes):
#         super().__init__()
#         self.features = nn.Sequential(
#             nn.Conv2d(3, ???, kernel_size=3, padding="same"),
#             nn.BatchNorm2d(???),
#             nn.ReLU(),
#             ...
#         )
#         ...


## Task 3: Model Training
Train your model and explain how you trained it. This includes:

- Data preprocessing
- [Data augmentation](https://medium.com/@saiwadotai/the-essential-guide-to-data-augmentation-in-deep-learning-f66e0907cdc8)?
- Train/validation split
- Choice of optimizer and its hyperparameters, including

  - learning rate
  - number of training epochs
  - batch size

We have not covered all of the above in the lectures yet, but we will get there soon.

Here is a template that you can use (again, gaps are marked with `???`).

**Note:** Formally, you should never touch your test set until you are completely done training and tuning your model. Below we carve out a small **validation split** from the *training* set using `random_split`, and only use the test set once, at the very end, in Task 4.

**Tip:** If training loss doesn't decrease at all after a few epochs, double check your learning rate isn't too high, and that you're calling `optimizer.zero_grad()` before every `loss.backward()`.

In [ ]:
# Data augmentation (optional). Set data_augmentation = True to enable it.
# QUESTION: in datasets like CIFAR-10, we can use RandomHorizontalFlip to augment the data.
# Why not here?
data_augmentation = ???

if data_augmentation:
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
    ])
else:
    train_transform = transforms.ToTensor()

# Re-load the training set with the transform above (the test set stays untouched: transforms.ToTensor() only).
train_dataset_aug = torchvision.datasets.SVHN(root="./data", split="train", download=True, transform=train_transform)

# Carve out a validation split from the training data (see note above -- do NOT use the test set for this).
val_fraction = 0.1
num_val = int(len(train_dataset_aug) * val_fraction)
num_train = len(train_dataset_aug) - num_val
train_subset, val_subset = random_split(
    train_dataset_aug, [num_train, num_val], generator=torch.Generator().manual_seed(0)
)
print(f"Train: {len(train_subset)}, Validation: {len(val_subset)}, Test: {len(test_dataset)}")


In [ ]:
learning_rate = ???
batch_size = ???
epochs = ???

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


def evaluate(model, loader):
    '''Returns (average loss, accuracy) of `model` on the data in `loader`.'''
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += images.size(0)

    train_loss, train_acc = running_loss / total, correct / total
    val_loss, val_acc = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{epochs} - train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f} "
          f"- val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].set_title("Loss curves")

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].set_title("Accuracy curves")
plt.tight_layout()
plt.show()


Take a look at your loss curves above. Is your model overfitting, underfitting, or about right? (This connects directly to what you've seen in the lectures on detecting overfitting from loss curves.)

## Task 4: Summarizing your results
Summarize your results using
 - Test set error (top-1 and top-5)
 - Test set accuracy
 - Confusion matrix
 - Precision/recall curve
 - Average Precision

**IMPORTANT:** You must summarize your results based on the *entire* test set (i.e., all ~26,000 samples) — this is the one time we use it.

You might find the code below useful:

In [ ]:
model.eval()
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

all_labels = []
all_probs = []  # softmax probabilities, shape (N, num_classes)

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.cpu())
        all_labels.append(labels)

y_score = torch.cat(all_probs).numpy()   # shape (~26000, num_classes)
y_true = torch.cat(all_labels).numpy()   # shape (~26000,)

print("y_score shape:", y_score.shape)
print("y_true shape:", y_true.shape)


### 4.1 Test set accuracy
The **top-1 accuracy** is simply the number of correct predictions divided by the total number of observations (in the test set).

To calculate the **top-5 accuracy**, a prediction counts as correct if the true label is among the model's 5 highest-scoring classes (rather than only its single top prediction).

Use [`torchmetrics.classification.MulticlassAccuracy`](https://lightning.ai/docs/torchmetrics/stable/classification/accuracy.html) for both -- it takes the desired `top_k` directly, so you don't need to rank `y_score` by hand. (If you're curious how it works internally, it's equivalent to sorting each row of `y_score` -- e.g. with [`torch.topk`](https://pytorch.org/docs/stable/generated/torch.topk.html) -- and checking whether `y_true` appears among the top-k indices.)

In [ ]:
# Your code goes here
# Hint: MulticlassAccuracy(num_classes=num_classes, top_k=1, average="micro")(y_score, y_true)
# and the same again with top_k=5


### 4.2 Test set error
Is just 1 minus the top-1 accuracy (top-1 error) or 1 minus the top-5 accuracy (top-5 error).


In [ ]:
# Your code goes here


### 4.3 Confusion matrix
A confusion matrix is a table that summarizes the performance of a classification model on a set of test data for which the true labels are known. The number of correct and incorrect predictions is summarized and broken down for each label. The diagonal elements of the table (from top-left to bottom-right) represent the number of correct predictions for each label. The off-diagonal elements correspond to incorrect predictions; they show the ways in which the classification model is confused when it makes predictions.

Example:

![alt text](https://scikit-learn.org/stable/_images/sphx_glr_plot_confusion_matrix_001.png)

You can normalize the entries of the table by dividing the numbers in each row by the sum of the numbers in that row. As a general rule of thumb, a score above 0.8 on the diagonal is desired.

Example:

![alt text](https://scikit-learn.org/stable/_images/sphx_glr_plot_confusion_matrix_002.png)

Calculate and display the normalized confusion matrix using [`torchmetrics.classification.MulticlassConfusionMatrix`](https://lightning.ai/docs/torchmetrics/stable/classification/confusion_matrix.html) -- it has a built-in `.plot()` method, so no manual matplotlib plumbing is needed. (If you'd rather compute it with scikit-learn instead, [`sklearn.metrics.confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) works too -- either is fine, but you'll use `torchmetrics` again in Labs 4b and 4c, so it's worth getting comfortable with it now.)


In [ ]:
# Your code goes here
# Hint: MulticlassConfusionMatrix(num_classes=num_classes, normalize="true"), then
# .update(y_score, y_true) and .plot(labels=class_names)


Take a look at your confusion matrix. Which digits get confused with each other most often? Does that make intuitive sense given what the digits look like (e.g. visually similar pairs like 3/8, 4/9, or 1/7)? Note that with a small toy CNN, your specific confusions may well differ from what a larger model like a ResNet would produce — that's expected and worth commenting on in your report.

### 4.4 Precision-Recall and Average Precision
Precision-Recall is a useful measure of the success of prediction, especially when the classes are very imbalanced. In information retrieval, precision is a measure of result relevancy:

```
precision = #TP/(#TP + #FP)
```

while recall is a measure of how many truly relevant results are returned:

```
recall = #TP/(#TP + #FN)
```

The precision-recall curve shows the tradeoff between precision and recall for different thresholds. A high area under the curve represents both high recall and high precision, where high precision relates to a low false positive rate, and high recall relates to a low false negative rate. High scores for both show that the classifier is returning accurate results (high precision), as well as returning a majority of all positive results (high recall).

Your task is to compute and plot a **one-vs-rest precision-recall curve per class**, plus the **Average Precision (AP)** per class, using [`torchmetrics.classification.MulticlassPrecisionRecallCurve`](https://lightning.ai/docs/torchmetrics/stable/classification/precision_recall_curve.html) and [`MulticlassAveragePrecision`](https://lightning.ai/docs/torchmetrics/stable/classification/average_precision). Both operate directly on the `(N, num_classes)` softmax tensor `y_score` you already have -- no manual one-vs-rest loop needed. `MulticlassPrecisionRecallCurve` also has a built-in `.plot(score=True)` method.

**Bonus (optional):** [this scikit-learn tutorial](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html) computes the same thing a more manual way -- worth a look if you want to see precision-recall computed from first principles, or as a sanity check against your `torchmetrics` result.


In [ ]:
# Your code goes here
# Hint: MulticlassPrecisionRecallCurve(num_classes=num_classes), then .update(y_score, y_true) and .plot(score=True)
# Hint: MulticlassAveragePrecision(num_classes=num_classes, average=None) for per-class AP


See if you can make sense of the outputs and interpret the results. Which classes have the highest/lowest average precision, and does that match what you saw in the confusion matrix?

---

**Next up:** `Lab 4b` moves from classification to *segmentation*, using a pretrained model and a different pair of metrics: IoU and Dice.